In [1]:
from ultralytics import YOLO
import torch
from pathlib import Path

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA IS:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU!!!:", torch.cuda.get_device_name(0))
else:
    print("GPU NOT FOUND!!!")

PyTorch version: 2.5.1+cu121
CUDA IS: True
GPU!!!: NVIDIA GeForce RTX 4070


In [3]:
project_path = Path(r"C:\Users\misha\OneDrive\Desktop\DL\yolo-damage")
yaml_path = project_path / "data.yaml" 

print("\nProject path:", project_path)
print("data.yaml exists:", yaml_path.exists())


Project path: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage
data.yaml exists: True


In [5]:
model = YOLO("yolo11n.pt")

In [6]:
train_params = {
    "data": str(yaml_path),
    "epochs": 30,
    "imgsz": 1024, 
    "batch": 4,
    "project": str(project_path / "runs"),
    "name": "damage_yolo_run1",
    "device": 0
}

print("\nTrain parameters:")
for key, value in train_params.items():
    print(f"  {key}: {value}")


Train parameters:
  data: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml
  epochs: 30
  imgsz: 1024
  batch: 4
  project: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\runs
  name: damage_yolo_run1
  device: 0


In [7]:
results = model.train(**train_params)

Ultralytics 8.4.63  Python-3.12.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=damage_yolo_run1-3, nbs=64, nms=False, opset=None, optimize=False,

In [4]:
model = YOLO("yolo11s.pt")

train_params = {
    "data": str(yaml_path),
    "epochs": 100,
    "patience": 20,
    "imgsz": 640,
    "batch": 8,
    "project": str(project_path / "runs"),
    "name": "damage_yolo11s_100ep",
    "device": 0
}

torch.cuda.empty_cache()

print("\nStarting YOLO11s training...")
for key, value in train_params.items():
    print(f"  {key}: {value}")

results = model.train(**train_params)
print("Training completed.")


Starting YOLO11s training...
  data: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml
  epochs: 100
  patience: 20
  imgsz: 640
  batch: 8
  project: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\runs
  name: damage_yolo11s_100ep
  device: 0
Ultralytics 8.4.63  Python-3.12.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, i

Here I decided to increase picture quality while training and not compress images to 640 by 640, keeping 'em in the original 1024 resolution -> training time will increase and my GPU won't be able to handle batches too big, so I lowered the batch size to 4. Also, I decided to fix the data imbalance by adding heavy spatial augmentations to the original dataset and applying "cls_pw": 1.0!!!!! - this class weighting is hella important for an imbalanced dataset like ours.

In [5]:
model = YOLO("yolo11s.pt")

train_params3 = {
    "data": str(yaml_path),
    "epochs": 100,
    "patience": 20,
    "imgsz": 1024,
    "batch": 4,
    "degrees": 180.0,
    "flipud": 0.5,
    "fliplr": 0.5,
    "cls_pw": 1.0,
    "project": str(project_path / "runs"),
    "name": "damage_yolo11s_1024_balanced",
    "device": 0
}

torch.cuda.empty_cache()

print("Starting training with target parameters...")
for key, value in train_params3.items():
    print(f"  {key}: {value}")

results = model.train(**train_params3)
print("Training completed.")

Starting training with target parameters...
  data: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml
  epochs: 100
  patience: 20
  imgsz: 1024
  batch: 4
  degrees: 180.0
  flipud: 0.5
  fliplr: 0.5
  cls_pw: 1.0
  project: C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\runs
  name: damage_yolo11s_1024_balanced
  device: 0
Ultralytics 8.4.63  Python-3.12.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12281MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\misha\OneDrive\Desktop\DL\yolo-damage\data.yaml, degrees=180.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, 